In [3]:
import pandas as pd

file_name = r'D:\Aa中工互联\工作安排\英格索兰\code\预测\lstm\data\拟合数据.xlsx'
df = pd.read_excel(file_name)

In [5]:
df.head()

,date,Actual,predict
0,2025-05-01 06:00:00,124.435000,0.000000
1,2025-05-01 06:30:00,164.596333,217.761250
2,2025-05-01 07:00:00,181.732667,204.759356
3,2025-05-01 07:30:00,185.767000,198.870006
4,2025-05-01 08:00:00,199.575333,189.801906


In [7]:
# 确保时间列为 datetime 类型
df['date'] = pd.to_datetime(df['date'])

# 提取时间特征
df['hour'] = df['date'].dt.hour
df['weekday'] = df['date'].dt.weekday
df['day'] = df['date'].dt.day

In [9]:
# 残差
df['residual'] = df['Actual'] - df['predict']

In [10]:
# 加入三班倒特征
df['day_part'] = df['hour'].apply(
    lambda x: 0 if 0<=x<6 else 1 if 6<=x<11 else 2 if 11<=x<14 else 3 if 14<=x<22 else 4
)

In [12]:
# 滞后特征生成
for lag in [1, 2, 3, 24, 48]:  # 30min间隔：24=12小时前
    df[f'residual_lag{lag}'] = df['residual'].shift(lag)
    df[f'SARIMA_lag{lag}'] = df['predict'].shift(lag)

# 滚动统计量(8小时工作制)
df['rolling_8h_mean'] = df['residual'].rolling(16).mean()  # 8小时(16=8*2)平均值窗口

In [13]:
df.head()

,date,Actual,predict,hour,weekday,day,residual,day_part,residual_lag1,SARIMA_lag1,residual_lag2,SARIMA_lag2,residual_lag3,SARIMA_lag3,residual_lag24,SARIMA_lag24,residual_lag48,SARIMA_lag48,rolling_8h_mean
0,2025-05-01 06:00:00,124.435000,0.000000,6,3,1,124.435000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-05-01 06:30:00,164.596333,217.761250,6,3,1,-53.164917,1,124.435000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-05-01 07:00:00,181.732667,204.759356,7,3,1,-23.026690,1,-53.164917,217.761250,124.435000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-05-01 07:30:00,185.767000,198.870006,7,3,1,-13.103006,1,-23.026690,204.759356,-53.164917,217.761250,124.435000,0.00000,NaN,NaN,NaN,NaN,NaN
4,2025-05-01 08:00:00,199.575333,189.801906,8,3,1,9.773427,1,-13.103006,198.870006,-23.026690,204.759356,-53.164917,217.76125,NaN,NaN,NaN,NaN,NaN


In [14]:
df.dropna(inplace=True)

In [16]:
len(df.columns)

19

In [ ]:
df.to_excel('残差修正数据集.xlsx',index=False)
print('保存成功！')

保存成功！
